In [ ]:
!git clone https://github.com/vinhpt-ueh/fast-plate-ocr.git
%cd fast-plate-ocr
!git checkout old-MobileViTV2W050
!pip install .[all]

Cloning into 'fast-plate-ocr'...
remote: Enumerating objects: 1842, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 1842 (delta 44), reused 52 (delta 36), pack-reused 1773 (from 2)
Receiving objects: 100% (1842/1842), 273.07 MiB | 18.71 MiB/s, done.
Resolving deltas: 100% (971/971), done.
/kaggle/working/fast-plate-ocr
Branch 'MobileViTV2-backbone' set up to track remote branch 'MobileViTV2-backbone' from 'origin'.
Switched to a new branch 'MobileViTV2-backbone'
Processing /kaggle/working/fast-plate-ocr
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.4/123.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 88.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
!pip install albumentations==2.0.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.6/290.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 632.7/632.7 kB 39.9 MB/s eta 0:00:00
  Attempting uninstall: albucore
    Found existing installation: albucore 0.0.19
    Uninstalling albucore-0.0.19:
      Successfully uninstalled albucore-0.0.19
  Attempting uninstall: albumentations
    Found existing installation: albumentations 1.4.20
    Uninstalling albumentations-1.4.20:
      Successfully uninstalled albumentations-1.4.20


In [11]:
import albumentations as A

transform_pipeline = A.Compose(
    [
    A.Rotate(limit=15, p=0.5),
    ]
)

# Export to a file (this resultant YAML can be used by the train script)
A.save(transform_pipeline, "./custom_transform_pipeline.yaml", data_format="yaml")

In [ ]:
!KERAS_BACKEND=tensorflow python -m fast_plate_ocr.cli.train \
	--annotations /kaggle/input/ufpr-char-addition/train.csv \
	--val-annotations /kaggle/input/ufpr-char-addition/valid.csv \
	--config-file /kaggle/input/ufpr-char-addition/config.yaml \
	--batch-size 16 \
	--epochs 20 \
	--dense \
	--num-workers 0 \
	--tensorboard \
	--early-stopping-patience 20 \
	--reduce-lr-patience 10 \
	--lr 0.00005 \
  --no-use-stn \
  --label-smoothing 0.05 \
  --weights-path '/kaggle/working/fast-plate-ocr/base_model.keras'

2025-03-21 10:32:49.363978: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-21 10:32:49.385793: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-21 10:32:49.392449: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/usr/local/lib/python3.10/dist-packages/albumentations/core/validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/usr/local/lib/python3.10/dist-packages/fast_plate_ocr/train/data/augmentation.py:12: UserWarning: Argument(s) 'value' are not valid for tr

In [ ]:
# fast_plate_ocr export-onnx \
# 	--model /kaggle/working/fast-plate-ocr/trained_models/2025-03-20_03-15-22/cnn_ocr-epoch_17-acc_0.843.keras \
# 	--output-path /kaggle/working/fast-plate-ocr/trained_models/ufpr.onnx \
# 	--opset 18 \
# 	--config-file /kaggle/input/ufpr-char-dataset/config.yaml